In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import sqrt, isfinite
from scipy.stats import t as t_dist, spearmanr, norm

In [36]:
# Pandas :: view ALL
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [38]:
# Pandas :: Default view
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.width')
pd.reset_option('display.max_colwidth')

In [2]:
# Defines
def analyse_dataframe(df):
    nan_cnt = df.isnull().sum()
    nan_rto = df.isnull().mean()*100
    unique_vals_cnt = df.nunique()

    ana_result = pd.DataFrame({
        'NaN Counts' : nan_cnt
        , 'NaN Ratio (%)' : nan_rto
        , 'Unique Values' : unique_vals_cnt
    })
    return ana_result


In [3]:
## PRE
out = r"D:\JHAhn\files"

In [7]:
sheets = ["Sheet1", "Sheet2"]
dfs = {name: pd.read_excel(os.path.join(out, 'Table 3-4.xlsx')
                           , sheet_name=name) for name in sheets
      }
dfs

{'Sheet1':                    MRI measures                                  Unnamed: 1  \
 0                           NaN                                         NaN   
 1                    ALPS index                                  All (Left)   
 2                           NaN                                b=800 (Left)   
 3                           NaN                               b=2000 (Left)   
 4                           NaN                                 All (Right)   
 5                           NaN                               b=800 (Right)   
 6                           NaN                              b=2000 (Right)   
 7   Conductivity\n(HFC, EC, IC)                             Amygdala (Left)   
 8                           NaN                            Amygdala (Right)   
 9                           NaN                          Hippocampus (Left)   
 10                          NaN                         Hippocampus (Right)   
 11                          N

In [11]:
def expand_multiline_rows(df: pd.DataFrame) -> pd.DataFrame:
    # 문자열로 통일, 줄바꿈 정규화
    df = df.astype(str).applymap(lambda x: x.replace("\r\n","\n").replace("\r","\n"))
    rows = []
    for _, row in df.iterrows():
        split_cols = [c.split("\n") if c != "nan" else [""] for c in row]
        h = max(len(c) for c in split_cols)
        for i in range(h):
            new = [(c[i] if i < len(c) else "") for c in split_cols]
            # 공백/따옴표/≤ 뒤 공백 정리
            new = [x.strip().strip('"').replace("≤ ","≤") for x in new]
            rows.append(new)
    return pd.DataFrame(rows, columns=df.columns)

with pd.ExcelWriter(os.path.join(out, "Table 3-4_M01.xlsx")
                    , engine="openpyxl") as w:
    for sh in ("Sheet1","Sheet2"):
        df = pd.read_excel(os.path.join(out, "Table 3-4.xlsx")
                           , sheet_name=sh, dtype=str, keep_default_na=False)
        expand_multiline_rows(df).to_excel(w, sheet_name=sh, index=False)